1. buissness understanding

The company wants to determine suitable locations for drone depots
based on the geographic distribution of its customers.

2. Data Understanding

The dataset contains clientid, x and y coordinates for 5956 customers.
The x and y coordinates represent customer locations in a 2D plane.

3. Data Preparation

The clientid column is not used for clustering because it is an identifier.
Only x and y are used as clustering features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

df = pd.read_csv('drone_cust_locations.csv', sep=';')
print(df)

X = df.drop(columns='clientid')

df.plot.hexbin(x="x", y="y", gridsize=25)


Darker areas means more customers, light areas have less customers.  
x - The x coordinate of the customer's location, when plotted into a 2D plane  
y - The y coordinate of the customer's location, when plotted into a 2D plane

In [ ]:
from sklearn.cluster import KMeans

model = KMeans(init='random', n_clusters=3, random_state=69)

start_1 = time.perf_counter()
model.fit(X)
end_1 = time.perf_counter()

plot = X.plot.hexbin(x="x", y="y", gridsize=25);

centers = model.cluster_centers_
print(centers)
plot.scatter(centers[:, 0],centers[:, 1],color="red")
plt.show()

In [ ]:
#new df with new column with nearest center of current row as value

X_2 = X.copy()
X_2["depot"] = model.labels_
X_2["depot"] = X_2["depot"].astype("category")

print(X_2.head(10))

In [ ]:
plot_2 = X_2.plot.scatter(x="x", y="y", c="depot",cmap="tab10", s=20);
plot_2.scatter(centers[:, 0],centers[:, 1],color="black",s=50)
plt.show()

In [ ]:
model_2 = KMeans(init='random', n_clusters=10, random_state=69)

start_2 = time.perf_counter()
model_2.fit(X)
end_2 = time.perf_counter()

X_3 = X.copy()
X_3["depot"] = model_2.labels_
X_3["depot"] = X_3["depot"].astype("category")

plot_3 = X_3.plot.scatter(x="x", y="y", c="depot",cmap="tab10", s=20);
plot_3.scatter(model_2.cluster_centers_[:, 0],model_2.cluster_centers_[:, 1],color="black",s=50)
plt.show()

print(f"model 1: {(end_1 - start_1) * 1000:.3f} ms | model 2: {(end_2 - start_2) * 1000:.3f} ms")

no significant difference in computation time

In [ ]:
wcss = []
for i in range(1,30):
    model = KMeans(init='random', n_clusters=i, random_state=42).fit(X)
    wcss.append(model.inertia_)
    
plt.plot(range(1,30), wcss, 'o-')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

agglomerative:

In [ ]:
from sklearn.cluster import AgglomerativeClustering

for k in range(2, 20):
    
    model_3 = AgglomerativeClustering(n_clusters=k)
    labels = model_3.fit_predict(X)

    plt.scatter(X["x"], X["y"], c=labels, cmap="tab10")
    centers = X.groupby(labels).mean()

    plt.scatter(
        centers["x"],
        centers["y"],
        color="black",
        s=50
    )
    
    plt.title(f"agglomerative Clustering - {k} Depots")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.show()

The agglomerative clustering produces plots that are less uniform or have jagged edges and the k mean algorithm produces plots that have more straight edges.